# Seurat to AnnData

Python-first notebook entry point for converting the Shi paper-QC, Varela DIV30, and Varela DIV90 Seurat objects into cached AnnData `.h5ad` files.

Run this with the `Python (mge-organoid-python)` kernel. Set `PROJECT_ROOT` before launching Jupyter:

```bash
export PROJECT_ROOT=/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder
```

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidate_roots = [cwd, cwd.parent]
if len(cwd.parents) >= 2:
    candidate_roots.append(cwd.parents[1])

for root in candidate_roots:
    src = root / "python_notebooks" / "src"
    if src.exists():
        sys.path.insert(0, str(src))
        repo_root = root
        break
else:
    raise RuntimeError("Could not locate python_notebooks/src from the current notebook directory")

repo_root

PosixPath('/home/elcrespo/Desktop/githubprojects/mge_organoid_pipeline')

In [2]:
from mge_organoid_python import (
    SeuratToAnnDataConverter,
    default_studies,
    resolve_project_root,
    validate_source_paths,
)

project_root = resolve_project_root()
studies = default_studies()

print(f"PROJECT_ROOT = {project_root}")
for study in studies:
    print(f"{study.study_id}: {study.seurat_path}")

PROJECT_ROOT = /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder
shi_2019_paper_qc: /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/results/shi_2019_paper_qc/shi_2019_seurat.rds
varela_div30: /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/results/varela_this_paper/varela_this_paper_seurat.rds
varela_div90: /nfs/turbo/umms-parent/Manny_test/ventral_sosrs_output/umap_props_output/clustered_day90_with_cluster_names_2.rds


In [4]:
missing = validate_source_paths(studies)
if missing:
    for study_id, path in missing:
        print(f"MISSING {study_id}: {path}")
    raise FileNotFoundError("One or more canonical Seurat inputs are missing")

print("All canonical Seurat inputs exist.")

All canonical Seurat inputs exist.


In [5]:
import os
import platform
import shutil
import sys
from pathlib import Path

print("Resource diagnostics before conversion")
hostname = platform.node()
project_root_env = os.environ.get("PROJECT_ROOT")
rscript = shutil.which("Rscript")

print("hostname:", hostname)
print("python:", sys.executable)
print("PROJECT_ROOT:", project_root_env)
print("Rscript:", rscript)
for key in ["SLURM_JOB_ID", "SLURM_JOB_NODELIST", "SLURM_CPUS_PER_TASK", "SLURM_MEM_PER_NODE"]:
    print(f"{key}:", os.environ.get(key))

print("CPU count:", os.cpu_count())
meminfo = Path("/proc/meminfo").read_text().splitlines()[:5]
print("/proc/meminfo first lines:")
print("\n".join(meminfo))

if hostname.startswith("gl-login"):
    raise RuntimeError(
        f"Notebook kernel is running on login node {hostname}. "
        "Do not run conversion here. Connect VS Code/Jupyter to the allocated compute node first."
    )
if not project_root_env:
    raise RuntimeError(
        "PROJECT_ROOT is not set in this notebook kernel. Run: "
        "export PROJECT_ROOT=/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder "
        "before launching/attaching the notebook kernel."
    )
if rscript is None:
    raise RuntimeError("Rscript is not available in this notebook kernel environment.")


Resource diagnostics before conversion
hostname: gl3112.arc-ts.umich.edu
python: /home/elcrespo/miniconda3/envs/mge-organoid-python/bin/python
PROJECT_ROOT: /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder
Rscript: /home/elcrespo/miniconda3/envs/mge-organoid-python/bin/Rscript
SLURM_JOB_ID: 50290604
SLURM_JOB_NODELIST: gl3112
SLURM_CPUS_PER_TASK: 4
SLURM_MEM_PER_NODE: 131072
CPU count: 36
/proc/meminfo first lines:
MemTotal:       195975448 kB
MemFree:        157262976 kB
MemAvailable:   184826732 kB
Buffers:             672 kB
Cached:         28103836 kB


In [6]:
converter = SeuratToAnnDataConverter(project_root=project_root)
print(f"AnnData cache directory: {converter.output_dir}")

for study in studies:
    print(study.study_id, "->", converter.output_path(study), "needs_conversion=", converter.needs_conversion(study))

AnnData cache directory: /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/results/python_anndata
shi_2019_paper_qc -> /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/results/python_anndata/shi_2019_paper_qc.h5ad needs_conversion= True
varela_div30 -> /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/results/python_anndata/varela_div30.h5ad needs_conversion= True
varela_div90 -> /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/results/python_anndata/varela_div90.h5ad needs_conversion= True


The next cell performs conversion. For large Seurat objects, run it in an interactive compute or GUI session rather than using login-node resources for long jobs.

In [7]:
import pandas as pd
from IPython.display import display

studies_by_id = {study.study_id: study for study in studies}
study = studies_by_id["shi_2019_paper_qc"]

print("Smoke test: converting/loading one small study first.", flush=True)
print("Study:", study.study_id, flush=True)
print("Source:", study.seurat_path, flush=True)
print("Target:", converter.output_path(study), flush=True)
print("needs_conversion=", converter.needs_conversion(study), flush=True)

adata, report = converter.convert(study)
adatas = {study.study_id: adata}
reports = [report]
reports_df = pd.DataFrame([report.as_dict()])
print("Smoke test complete.", flush=True)
display(reports_df)


Smoke test: converting/loading one small study first.
Study: shi_2019_paper_qc
Source: /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/results/shi_2019_paper_qc/shi_2019_seurat.rds
Target: /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/results/python_anndata/shi_2019_paper_qc.h5ad
needs_conversion= True
[2026-05-15 15:19:44] Study shi_2019_paper_qc: source=/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/results/shi_2019_paper_qc/shi_2019_seurat.rds target=/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/results/python_anndata/shi_2019_paper_qc.h5ad needs_conversion=True
[2026-05-15 15:19:44] Study shi_2019_paper_qc: starting RDS -> H5AD conversion
[2026-05-15 15:19:44] Running Rscript subprocess: Rscript /home/elcrespo/Desktop/githubprojects/mge_organoid_pipeline/python_notebooks/scripts/seurat_to_h5ad.R --seurat /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/results/shi_2019_paper_qc/shi_2019_seurat.rds --h5ad /nfs/turbo/umms-parent/m

KeyboardInterrupt: 

In [ ]:
# Optional: after the Shi smoke test succeeds, convert large Varela studies one at a time.
# This writes .h5ad files but does not load them into memory.
# Run one line at a time, not both at once, if you want the safest memory behavior.

# varela_div30_path = converter.convert_file(studies_by_id["varela_div30"])
# print("Varela DIV30 cached at:", varela_div30_path)

# varela_div90_path = converter.convert_file(studies_by_id["varela_div90"])
# print("Varela DIV90 cached at:", varela_div90_path)


In [ ]:
import matplotlib.pyplot as plt

for study_id, adata in adatas.items():
    umap = adata.obsm["X_umap"]
    fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
    ax.scatter(umap[:, 0], umap[:, 1], s=1, linewidths=0, alpha=0.6)
    ax.set_title(f"{study_id}\nn={adata.n_obs:,}")
    ax.set_xlabel("UMAP_1")
    ax.set_ylabel("UMAP_2")
    plt.show()


In [ ]:
# Access the loaded smoke-test AnnData object.
shi = adatas["shi_2019_paper_qc"]
shi
